# 🛡️ CPG-Based Malware Classification with HGT

**Unified Instruction Representation (UIR)** — Train, evaluate, save, and visualize a Heterogeneous Graph Transformer (HGT) model for malware detection using Code Property Graphs (CPGs).

### Features
- ✅ **Train / Validation / Test Split** (60% / 20% / 20%)
- ✅ **Early Stopping** with configurable patience & min-delta
- ✅ **LR Scheduler** — ReduceLROnPlateau (adapts to validation loss)
- ✅ **Best Weights Restore** — automatically loads best checkpoint after training
- ✅ **Test Set Evaluation** — final metrics on unseen held-out test data

---

## Table of Contents
1. [Setup & Imports](#1.-Setup-&-Imports)
2. [Configuration](#2.-Configuration)
3. [Data Loading & Exploration](#3.-Data-Loading-&-Exploration)
4. [Dataset Statistics & Visualization](#4.-Dataset-Statistics-&-Visualization)
5. [Train / Validation / Test Split](#5.-Train-/-Validation-/-Test-Split)
6. [Model Initialization](#6.-Model-Initialization)
7. [Early Stopping & LR Scheduler](#7.-Early-Stopping-&-LR-Scheduler)
8. [Training](#8.-Training)
9. [Test Set Evaluation](#9.-Test-Set-Evaluation)
10. [Save Model](#10.-Save-Model)
11. [Plots & Visualization](#11.-Plots-&-Visualization)

In [1]:
import os, sys, shutil

# Copy project files to a writable directory (Kaggle input is read-only)
INPUT_DIR = '/kaggle/input/datasets/sasindumalhara/cpg-malware-dataset2-0'
WORK_DIR  = '/kaggle/working/CPG'

if not os.path.exists(WORK_DIR):
    shutil.copytree(INPUT_DIR, WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Install missing dependencies
!pip install -q pydantic pylnk3

print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

Working directory: /kaggle/working/CPG
Files: ['setup.py', 'requirements.txt', 'uir', 'cpgs']


---
## 1. Setup & Imports

In [2]:
import sys, os, json, time, copy, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

warnings.filterwarnings('ignore')

# ─── Project imports ───────────────────────────────────────────────
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from uir.model.hgt import HeterogeneousGraphTransformer
from uir.model.dataset import CPGDataset, CPGData, collate_cpg_batch
from uir.model.trainer import Trainer, ContrastiveLoss
from uir.model.evaluator import Evaluator, compute_roc_auc
from uir.cpg.graph import CodePropertyGraph
from uir.cpg.schema import NodeType, EdgeType
from uir.config import ModelConfig, TrainingConfig

# ─── Plotting defaults ─────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
matplotlib.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'PyTorch {torch.__version__}')
print(f'Device : {DEVICE}')
print(f'Project: {PROJECT_ROOT}')

PyTorch 2.8.0+cu126
Device : cuda
Project: /kaggle/working/CPG


---
## 2. Configuration

In [3]:
# ─── Hyperparameters ───────────────────────────────────────────────
CPG_DIR        = Path('/kaggle/working/CPG/cpgs')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Model
INPUT_DIM       = 256
HIDDEN_DIM      = 256
NUM_LAYERS      = 4
NUM_HEADS       = 8
NUM_CLASSES     = 2
DROPOUT         = 0.1
NUM_NODE_TYPES  = len(NodeType)
NUM_EDGE_TYPES  = len(EdgeType)

# Training
BATCH_SIZE      = 4
LEARNING_RATE   = 1e-4
NUM_EPOCHS      = 25
RANDOM_STATE    = 42

# Data split ratios (must sum to 1.0)
TRAIN_RATIO     = 0.8
VAL_RATIO       = 0.1
TEST_RATIO      = 0.1

# Early Stopping
ES_PATIENCE     = 8
ES_MIN_DELTA    = 0.001
ES_MONITOR      = 'val_loss'

# LR Scheduler (ReduceLROnPlateau)
LR_FACTOR       = 0.5
LR_PATIENCE     = 5
LR_MIN          = 1e-7

print('─' * 55)
print(f'CPG directory   : {CPG_DIR}')
print(f'Checkpoint dir  : {CHECKPOINT_DIR}')
print(f'Model dims      : input={INPUT_DIM}, hidden={HIDDEN_DIM}')
print(f'Layers / Heads  : {NUM_LAYERS} / {NUM_HEADS}')
print(f'Node/Edge types : {NUM_NODE_TYPES} / {NUM_EDGE_TYPES}')
print(f'Epochs / Batch  : {NUM_EPOCHS} / {BATCH_SIZE}')
print(f'Learning Rate   : {LEARNING_RATE}')
print(f'Split           : train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}')
print(f'Early Stop      : patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}, monitor={ES_MONITOR}')
print(f'LR Scheduler    : ReduceLROnPlateau(factor={LR_FACTOR}, patience={LR_PATIENCE}, min_lr={LR_MIN})')
print('─' * 55)

───────────────────────────────────────────────────────
CPG directory   : /kaggle/working/CPG/cpgs
Checkpoint dir  : /kaggle/working/checkpoints
Model dims      : input=256, hidden=256
Layers / Heads  : 4 / 8
Node/Edge types : 11 / 8
Epochs / Batch  : 1 / 4
Learning Rate   : 0.0001
Split           : train=0.8, val=0.1, test=0.1
Early Stop      : patience=10, min_delta=0.001, monitor=val_loss
LR Scheduler    : ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-07)
───────────────────────────────────────────────────────


---
## 3. Data Loading & Exploration

In [8]:
dataset = CPGDataset(CPG_DIR, embedding_dim=INPUT_DIM)
print(f'Total CPG files found: {len(dataset)}')

labels = []
node_counts = []
edge_counts = []
file_paths  = []

print('\nLoading graph statistics...')
for i in range(len(dataset)):
    data = dataset[i]
    labels.append(data.y.item())
    node_counts.append(data.num_nodes)
    edge_counts.append(data.num_edges)
    file_paths.append(data.file_path)

labels = np.array(labels)
node_counts = np.array(node_counts)
edge_counts = np.array(edge_counts)

n_benign  = (labels == 0).sum()
n_malware = (labels == 1).sum()

print(f'\n📊 Class Distribution:')
print(f'   Benign  (0): {n_benign}')
print(f'   Malware (1): {n_malware}')
print(f'   Total      : {len(labels)}')
print(f'\n📈 Graph sizes:')
print(f'   Nodes — min: {node_counts.min()}, max: {node_counts.max()}, '
      f'mean: {node_counts.mean():.1f}, median: {np.median(node_counts):.0f}')
print(f'   Edges — min: {edge_counts.min()}, max: {edge_counts.max()}, '
      f'mean: {edge_counts.mean():.1f}, median: {np.median(edge_counts):.0f}')

Total CPG files found: 1905

Loading graph statistics...

📊 Class Distribution:
   Benign  (0): 905
   Malware (1): 1000
   Total      : 1905

📈 Graph sizes:
   Nodes — min: 2, max: 10000, mean: 5986.0, median: 6070
   Edges — min: 0, max: 17404, mean: 9110.2, median: 9332


---
## 4. Dataset Statistics & Visualization

In [ ]:
"""
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Benign', 'Malware'], [n_benign, n_malware], color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, [n_benign, n_malware]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_title('Class Distribution')
ax.set_ylabel('Number of Samples')

ax = axes[0, 1]
ax.hist(node_counts[labels == 0], bins=20, alpha=0.65, label='Benign', color=colors[0], edgecolor='white')
ax.hist(node_counts[labels == 1], bins=20, alpha=0.65, label='Malware', color=colors[1], edgecolor='white')
ax.set_title('Node Count Distribution')
ax.set_xlabel('Number of Nodes'); ax.set_ylabel('Frequency'); ax.legend()

ax = axes[1, 0]
ax.hist(edge_counts[labels == 0], bins=20, alpha=0.65, label='Benign', color=colors[0], edgecolor='white')
ax.hist(edge_counts[labels == 1], bins=20, alpha=0.65, label='Malware', color=colors[1], edgecolor='white')
ax.set_title('Edge Count Distribution')
ax.set_xlabel('Number of Edges'); ax.set_ylabel('Frequency'); ax.legend()

ax = axes[1, 1]
ax.scatter(node_counts[labels == 0], edge_counts[labels == 0],
           alpha=0.7, c=colors[0], label='Benign', s=60, edgecolors='white', linewidth=0.5)
ax.scatter(node_counts[labels == 1], edge_counts[labels == 1],
           alpha=0.7, c=colors[1], label='Malware', s=60, edgecolors='white', linewidth=0.5)
ax.set_title('Nodes vs Edges')
ax.set_xlabel('Node Count'); ax.set_ylabel('Edge Count'); ax.legend()

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'dataset_statistics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/dataset_statistics.png')


"""

In [6]:
"""

print('Scanning node/edge type distributions across all graphs...\n')

node_type_counts_benign  = Counter()
node_type_counts_malware = Counter()
edge_type_counts_benign  = Counter()
edge_type_counts_malware = Counter()

for i in range(len(dataset)):
    cpg_path = dataset.cpg_files[i]
    try:
        cpg = CodePropertyGraph.load(cpg_path)
    except Exception:
        continue
    is_malware = labels[i] == 1
    for node in cpg.nodes.values():
        (node_type_counts_malware if is_malware else node_type_counts_benign)[node.node_type.value] += 1
    for edge in cpg.edges:
        (edge_type_counts_malware if is_malware else edge_type_counts_benign)[edge.edge_type.value] += 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
w = 0.35

all_nt = sorted(set(list(node_type_counts_benign) + list(node_type_counts_malware)))
x = np.arange(len(all_nt))
axes[0].bar(x - w/2, [node_type_counts_benign.get(t, 0) for t in all_nt], w, label='Benign', color='#2ecc71', edgecolor='white')
axes[0].bar(x + w/2, [node_type_counts_malware.get(t, 0) for t in all_nt], w, label='Malware', color='#e74c3c', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(all_nt, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Node Type Distribution'); axes[0].set_ylabel('Count'); axes[0].legend()

all_et = sorted(set(list(edge_type_counts_benign) + list(edge_type_counts_malware)))
x = np.arange(len(all_et))
axes[1].bar(x - w/2, [edge_type_counts_benign.get(t, 0) for t in all_et], w, label='Benign', color='#2ecc71', edgecolor='white')
axes[1].bar(x + w/2, [edge_type_counts_malware.get(t, 0) for t in all_et], w, label='Malware', color='#e74c3c', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(all_et, rotation=45, ha='right', fontsize=9)
axes[1].set_title('Edge Type Distribution'); axes[1].set_ylabel('Count'); axes[1].legend()

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'type_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/type_distributions.png')


"""

Scanning node/edge type distributions across all graphs...



KeyboardInterrupt: 

---
## 5. Train / Validation / Test Split

Stratified 3-way split:
- **Train (60%)** — model learns from this
- **Validation (20%)** — used for early stopping & LR scheduling during training
- **Test (20%)** — completely held out, used only for final evaluation

In [9]:
indices = list(range(len(dataset)))

# Step 1: split off the test set
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_RATIO,
    stratify=labels,
    random_state=RANDOM_STATE
)

# Step 2: split the remaining into train and validation
# val_ratio relative to train+val = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
relative_val_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_val_labels   = labels[train_val_idx]

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=relative_val_ratio,
    stratify=train_val_labels,
    random_state=RANDOM_STATE
)

# Create subsets
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

train_labels = labels[train_idx]
val_labels   = labels[val_idx]
test_labels  = labels[test_idx]

print('┌─────────────┬─────────┬─────────┬─────────┐')
print('│   Split     │  Total  │  Benign │ Malware │')
print('├─────────────┼─────────┼─────────┼─────────┤')
print(f'│ Train  {TRAIN_RATIO:.0%} │  {len(train_idx):5d}  │  {sum(train_labels==0):5d}  │  {sum(train_labels==1):5d}  │')
print(f'│ Val    {VAL_RATIO:.0%}  │  {len(val_idx):5d}  │  {sum(val_labels==0):5d}  │  {sum(val_labels==1):5d}  │')
print(f'│ Test   {TEST_RATIO:.0%}  │  {len(test_idx):5d}  │  {sum(test_labels==0):5d}  │  {sum(test_labels==1):5d}  │')
print('└─────────────┴─────────┴─────────┴─────────┘')

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_cpg_batch)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_cpg_batch)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_cpg_batch)

print(f'\nDataLoaders created: train={len(train_loader)} batches, '
      f'val={len(val_loader)} batches, test={len(test_loader)} batches')

┌─────────────┬─────────┬─────────┬─────────┐
│   Split     │  Total  │  Benign │ Malware │
├─────────────┼─────────┼─────────┼─────────┤
│ Train  80% │   1523  │    723  │    800  │
│ Val    10%  │    191  │     91  │    100  │
│ Test   10%  │    191  │     91  │    100  │
└─────────────┴─────────┴─────────┴─────────┘

DataLoaders created: train=381 batches, val=48 batches, test=48 batches


---
## 6. Model Initialization

In [10]:
model = HeterogeneousGraphTransformer(
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
    num_node_types=NUM_NODE_TYPES, num_edge_types=NUM_EDGE_TYPES,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
    num_classes=NUM_CLASSES, dropout=DROPOUT,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('═' * 50)
print('HGT Model Summary')
print('═' * 50)
print(f'  Input / Hidden : {INPUT_DIM} / {HIDDEN_DIM}')
print(f'  Layers / Heads : {NUM_LAYERS} / {NUM_HEADS}')
print(f'  Node / Edge    : {NUM_NODE_TYPES} / {NUM_EDGE_TYPES}')
print(f'  Classes        : {NUM_CLASSES}')
print(f'  Total params   : {total_params:,}')
print(f'  Trainable      : {trainable:,}')
print(f'  Device         : {DEVICE}')
print('═' * 50)

══════════════════════════════════════════════════
HGT Model Summary
══════════════════════════════════════════════════
  Input / Hidden : 256 / 256
  Layers / Heads : 4 / 8
  Node / Edge    : 11 / 8
  Classes        : 2
  Total params   : 13,358,595
  Trainable      : 13,358,595
  Device         : cuda
══════════════════════════════════════════════════


---
## 7. Early Stopping & LR Scheduler

In [11]:
class EarlyStopping:
    """Early stopping with in-memory best weights restore."""

    def __init__(self, patience=10, min_delta=0.001, monitor='val_loss', verbose=True):
        self.patience  = patience
        self.min_delta = min_delta
        self.monitor   = monitor
        self.verbose   = verbose
        self.counter      = 0
        self.best_score   = None
        self.best_epoch   = 0
        self.early_stop   = False
        self.best_weights = None
        self._mode = 'min' if 'loss' in monitor else 'max'

    def __call__(self, score, epoch, model):
        score_adj = -score if self._mode == 'min' else score

        if self.best_score is None:
            self.best_score = score_adj
            self.best_epoch = epoch
            self._save_weights(model)
            return

        if score_adj > self.best_score + self.min_delta:
            if self.verbose:
                raw_old = -self.best_score if self._mode == 'min' else self.best_score
                raw_new = -score_adj if self._mode == 'min' else score_adj
                print(f'  ↳ EarlyStopping: {self.monitor} improved '
                      f'{raw_old:.4f} → {raw_new:.4f}  ★ saving weights')
            self.best_score = score_adj
            self.best_epoch = epoch
            self.counter    = 0
            self._save_weights(model)
        else:
            self.counter += 1
            if self.verbose:
                print(f'  ↳ EarlyStopping: no improvement ({self.counter}/{self.patience})')
            if self.counter >= self.patience:
                self.early_stop = True

    def _save_weights(self, model):
        self.best_weights = copy.deepcopy(model.state_dict())

    def restore_best_weights(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)
            print(f'\n🔄 Restored best weights from epoch {self.best_epoch}')
        else:
            print('⚠️  No best weights saved — using current weights')


# ─── Instantiate ───────────────────────────────────────────────────
early_stopper = EarlyStopping(
    patience=ES_PATIENCE, min_delta=ES_MIN_DELTA,
    monitor=ES_MONITOR, verbose=True
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=LR_FACTOR,
    patience=LR_PATIENCE, min_lr=LR_MIN
)

ce_loss_fn   = nn.CrossEntropyLoss()
cont_loss_fn = ContrastiveLoss(temperature=0.07)

print('✅ Early stopping  : ready')
print('✅ LR scheduler    : ReduceLROnPlateau')
print('✅ Optimizer       : AdamW')
print('✅ Loss            : CrossEntropy + Contrastive')

✅ Early stopping  : ready
✅ LR scheduler    : ReduceLROnPlateau
✅ Optimizer       : AdamW
✅ Loss            : CrossEntropy + Contrastive


---
## 8. Training

- Uses **train set** for learning
- Uses **validation set** for early stopping & LR scheduling
- **Test set is NOT touched** during training

In [12]:
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'lr': []
}

print(f'Starting training for up to {NUM_EPOCHS} epochs...')
print(f'Train on {len(train_idx)} samples, validate on {len(val_idx)} samples')
print(f'Test set ({len(test_idx)} samples) held out for final evaluation\n')
print(f'{"Epoch":>7} │ {"Train Loss":>10} {"Train Acc":>10} │ '
      f'{"Val Loss":>10} {"Val Acc":>10} │ {"LR":>10}')
print('─' * 80)

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ── TRAIN ──────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_data, batch_idx in train_loader:
        batch_data = batch_data.to(DEVICE)
        batch_idx  = batch_idx.to(DEVICE)
        optimizer.zero_grad()

        logits = model(
            batch_data.x, batch_data.edge_index,
            batch_data.node_types, batch_data.edge_types, batch_idx
        )

        batch_labels = batch_data.y.view(-1)
        if batch_labels.numel() == 0:
            continue

        loss = ce_loss_fn(logits, batch_labels)

        embeddings = model.get_graph_embedding(
            batch_data.x, batch_data.edge_index,
            batch_data.node_types, batch_data.edge_types, batch_idx
        )
        loss = loss + 0.5 * cont_loss_fn(embeddings, batch_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        train_total   += batch_labels.numel()

    avg_train_loss = train_loss / max(len(train_loader), 1)
    avg_train_acc  = train_correct / max(train_total, 1)

    # ── VALIDATE ───────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch_data, batch_idx in val_loader:
            batch_data = batch_data.to(DEVICE)
            batch_idx  = batch_idx.to(DEVICE)

            logits = model(
                batch_data.x, batch_data.edge_index,
                batch_data.node_types, batch_data.edge_types, batch_idx
            )

            batch_labels = batch_data.y.view(-1)
            if batch_labels.numel() == 0:
                continue

            val_loss    += ce_loss_fn(logits, batch_labels).item()
            val_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            val_total   += batch_labels.numel()

    avg_val_loss = val_loss / max(len(val_loader), 1)
    avg_val_acc  = val_correct / max(val_total, 1)
    current_lr   = optimizer.param_groups[0]['lr']

    # Record history
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(avg_val_acc)
    history['lr'].append(current_lr)

    # ── LOG ─────────────────────────────────────────────────────────
    print(f'{epoch:3d}/{NUM_EPOCHS:3d} │ '
          f'{avg_train_loss:10.4f} {avg_train_acc:10.4f} │ '
          f'{avg_val_loss:10.4f} {avg_val_acc:10.4f} │ '
          f'{current_lr:10.2e}')

    # ── LR SCHEDULER ───────────────────────────────────────────────
    old_lr = current_lr
    lr_scheduler.step(avg_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < old_lr:
        print(f'  ↳ LR reduced: {old_lr:.2e} → {new_lr:.2e}')

    # ── EARLY STOPPING ─────────────────────────────────────────────
    es_score = avg_val_loss if ES_MONITOR == 'val_loss' else avg_val_acc
    early_stopper(es_score, epoch, model)

    if early_stopper.early_stop:
        print(f'\n⏹  Early stopping triggered at epoch {epoch}!')
        print(f'   Best epoch was {early_stopper.best_epoch}')
        break

elapsed = time.time() - start_time
print(f'\n✅ Training finished in {elapsed:.1f}s ({epoch} epochs)')

# ── RESTORE BEST WEIGHTS ───────────────────────────────────────────
early_stopper.restore_best_weights(model)

# Save best checkpoint to disk
torch.save({
    'epoch': early_stopper.best_epoch,
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'history': history,
}, CHECKPOINT_DIR / 'best_model.pt')
print(f'💾 Best model (epoch {early_stopper.best_epoch}) saved → checkpoints/best_model.pt')

Starting training for up to 1 epochs...
Train on 1523 samples, validate on 191 samples
Test set (191 samples) held out for final evaluation

  Epoch │ Train Loss  Train Acc │   Val Loss    Val Acc │         LR
────────────────────────────────────────────────────────────────────────────────
  1/  1 │     1.2730     0.6100 │     0.5702     0.7173 │   1.00e-04

✅ Training finished in 1037.8s (1 epochs)

🔄 Restored best weights from epoch 1
💾 Best model (epoch 1) saved → checkpoints/best_model.pt


---
## 9. Test Set Evaluation

Final evaluation on the **held-out test set** using **best weights** restored from training.

In [ ]:
model.eval()
print(f'Evaluating on TEST SET ({len(test_idx)} samples) '
      f'with best weights from epoch {early_stopper.best_epoch}\n')

all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for batch_data, batch_idx in test_loader:
        batch_data = batch_data.to(DEVICE)
        batch_idx  = batch_idx.to(DEVICE)

        logits = model(
            batch_data.x, batch_data.edge_index,
            batch_data.node_types, batch_data.edge_types, batch_idx
        )

        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)
        labels_batch = batch_data.y.view(-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels_batch.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

# ─── Metrics ───────────────────────────────────────────────────────
evaluator = Evaluator(num_classes=NUM_CLASSES)
metrics   = evaluator.compute_metrics(all_preds, all_labels)

print('═' * 55)
print('  TEST SET RESULTS (Best Weights — Unseen Data)')
print('═' * 55)
print(f'  Accuracy     : {metrics["accuracy"]:.4f}')
print(f'  Precision    : {metrics["precision"]:.4f}')
print(f'  Recall       : {metrics["recall"]:.4f}')
print(f'  F1 Score     : {metrics["f1"]:.4f}')
print(f'  FPR          : {metrics["false_positive_rate"]:.4f}')
print(f'  FNR          : {metrics["false_negative_rate"]:.4f}')
print('─' * 55)
print(f'  True Pos     : {metrics["true_positives"]}')
print(f'  True Neg     : {metrics["true_negatives"]}')
print(f'  False Pos    : {metrics["false_positives"]}')
print(f'  False Neg    : {metrics["false_negatives"]}')
print('═' * 55)

print('\n' + classification_report(all_labels, all_preds,
                                    target_names=['Benign', 'Malware']))

---
## 10. Save Model

In [ ]:
final_path = CHECKPOINT_DIR / 'final_model.pt'

torch.save({
    'model_state':    model.state_dict(),
    'model_config': {
        'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM,
        'num_node_types': NUM_NODE_TYPES, 'num_edge_types': NUM_EDGE_TYPES,
        'num_layers': NUM_LAYERS, 'num_heads': NUM_HEADS,
        'num_classes': NUM_CLASSES, 'dropout': DROPOUT,
    },
    'training_config': {
        'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'train_ratio': TRAIN_RATIO, 'val_ratio': VAL_RATIO, 'test_ratio': TEST_RATIO,
        'es_patience': ES_PATIENCE, 'es_min_delta': ES_MIN_DELTA,
        'lr_factor': LR_FACTOR, 'lr_patience': LR_PATIENCE,
    },
    'split_sizes': {
        'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx),
    },
    'best_epoch':      early_stopper.best_epoch,
    'test_metrics':    metrics,
    'history':         history,
}, final_path)

print(f'✅ Final model saved → {final_path}')
print(f'   Best epoch  : {early_stopper.best_epoch}')
print(f'   Test acc    : {metrics["accuracy"]:.4f}')
print(f'   File size   : {final_path.stat().st_size / 1024 / 1024:.2f} MB')

---
## 11. Plots & Visualization

In [ ]:
# ─── Training Curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history['train_loss']) + 1)
best_ep = early_stopper.best_epoch

# Loss
ax = axes[0]
ax.plot(ep, history['train_loss'], 'o-', label='Train', color='#3498db', lw=2, ms=4)
ax.plot(ep, history['val_loss'],   's-', label='Val',   color='#e74c3c', lw=2, ms=4)
ax.axvline(best_ep, color='green', ls='--', lw=1.5, alpha=0.7, label=f'Best (ep {best_ep})')
ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(ep, history['train_acc'], 'o-', label='Train', color='#3498db', lw=2, ms=4)
ax.plot(ep, history['val_acc'],   's-', label='Val',   color='#e74c3c', lw=2, ms=4)
ax.axvline(best_ep, color='green', ls='--', lw=1.5, alpha=0.7, label=f'Best (ep {best_ep})')
ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, alpha=0.3)

# Learning Rate
ax = axes[2]
ax.plot(ep, history['lr'], 'o-', color='#9b59b6', lw=2, ms=4)
ax.axvline(best_ep, color='green', ls='--', lw=1.5, alpha=0.7, label=f'Best (ep {best_ep})')
ax.set_title('Learning Rate (ReduceLROnPlateau)')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR'); ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/training_curves.png')

In [ ]:
# ─── Confusion Matrix (Test Set) ──────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'],
            square=True, linewidths=2, linecolor='white',
            annot_kws={'size': 18, 'weight': 'bold'}, ax=ax)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix (Test Set)', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/confusion_matrix.png')

In [ ]:
# ─── ROC & Precision-Recall Curves (Test Set) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc_val = auc(fpr, tpr)
ax = axes[0]
ax.plot(fpr, tpr, color='#e74c3c', lw=2.5, label=f'ROC (AUC={roc_auc_val:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.4)
ax.fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve (Test Set)')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

prec, rec, _ = precision_recall_curve(all_labels, all_probs)
pr_auc = auc(rec, prec)
ax = axes[1]
ax.plot(rec, prec, color='#3498db', lw=2.5, label=f'PR (AUC={pr_auc:.3f})')
ax.fill_between(rec, prec, alpha=0.15, color='#3498db')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Precision-Recall Curve (Test Set)')
ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/roc_pr_curves.png')

In [ ]:
# ─── Metrics Bar Chart (Test Set) ─────────────────────────────────
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metric_values = [metrics['accuracy'], metrics['precision'], metrics['recall'], metrics['f1']]

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(metric_names, metric_values, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_ylim(0, 1.15)
ax.set_title('Test Set Metrics', fontsize=15, fontweight='bold')
ax.set_ylabel('Score'); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/metrics_summary.png')

In [ ]:
# ─── Summary ──────────────────────────────────────────────────────
print('\n' + '═' * 60)
print('  TRAINING COMPLETE — SUMMARY')
print('═' * 60)
print(f'  Dataset        : {len(dataset)} CPGs ({n_benign} benign, {n_malware} malware)')
print(f'  Split          : train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')
print(f'  Model          : HGT ({total_params:,} params)')
print(f'  Best Epoch     : {early_stopper.best_epoch}')
print(f'  ─── Test Set Results ───')
print(f'  Accuracy       : {metrics["accuracy"]:.4f}')
print(f'  F1 Score       : {metrics["f1"]:.4f}')
print(f'  ROC-AUC        : {roc_auc_val:.4f}')
print(f'  PR-AUC         : {pr_auc:.4f}')
print(f'  ─── Settings ───')
print(f'  Early Stopping : patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}')
print(f'  LR Scheduler   : ReduceLROnPlateau(factor={LR_FACTOR}, patience={LR_PATIENCE})')
print(f'  Weights        : Best restored from epoch {early_stopper.best_epoch}')
print(f'\n  Saved files:')
print(f'    • checkpoints/best_model.pt')
print(f'    • checkpoints/final_model.pt')
print(f'    • checkpoints/training_curves.png')
print(f'    • checkpoints/confusion_matrix.png')
print(f'    • checkpoints/roc_pr_curves.png')
print(f'    • checkpoints/metrics_summary.png')
print(f'    • checkpoints/dataset_statistics.png')
print(f'    • checkpoints/type_distributions.png')
print('═' * 60)